<a href="https://colab.research.google.com/github/MarcoLuw/semal27-stereoqueereval/blob/main/experiments/StereoQueerEval_advanced_task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# StereoQueerEval - mBERT fine-tuning

Advanced model for the binary stereotype classification task (stereotype: yes / no).
This notebook fine-tunes a multilingual transformer encoder on YouTube comments across four languages (EN, IT, NL, FA).

The code is written to be reusable: to try another encoder later (XLM-R, etc.), change MODEL_NAME in the config cell. Nothing else needs to change.

## Run configuration

- Model: mBERT (bert-base-multilingual-cased)
- Input text: yt_comment (raw text, emojis and punctuation kept)
- Label: stereotype -> 1 if yes, else 0
- Split: 80 / 10 / 10 (train / validation / test), per language, stratified, seed 42
- Loss: plain cross-entropy (no class weighting for now)
- Max sequence length: 100 tokens
- Selection metric: macro-F1 (keep the best epoch, stop early if it stalls)

Baseline to beat (TF-IDF + Logistic Regression): macro-F1 ~ 0.66.

Note: the baseline used a different 80/20 split, so this is an approximate comparison, not a same-rows one.

## 0. Setup

In [1]:
# Install the training stack.
# Tip: once the environment works, pin exact versions here for reproducibility.
!pip install -q -U transformers datasets accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [2]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [3]:
# Configuration

# Model checkpoint. Change this to reuse the notebook for another encoder.
# mBERT cased: keeps letter case, which can carry signal in multilingual text.
MODEL_NAME = "bert-base-multilingual-cased"

# Input: which text fields feed the model
# Title and description can be added later by extending this list.
TEXT_COLUMNS = ["yt_comment"] # Comment only

# Label column and the positive class.
LABEL_COLUMN = "stereotype"
POSITIVE_LABEL = "yes"

# Max tokens per example. 100 fits comments (only ~9.4% exceed 70 words)
# Raise it later when long fields (description) are added
MAX_LENGTH = 100

# Split ratios (per language, stratified)
TRAIN_SIZE = 0.8
VAL_SIZE = 0.1
TEST_SIZE = 0.1

# Training hyperparameters
LEARNING_RATE = 2e-5
NUM_TRAIN_EPOCHS = 6          # early stopping usually stops before this
TRAIN_BATCH_SIZE = 32         # H100 can go higher
EVAL_BATCH_SIZE = 64          # batch size during evaluation/prediction
WEIGHT_DECAY = 0.01           # Similar idea to α (penalty) in regularization term of cost function
WARMUP_RATIO = 0.1            # warm up the learning rate over the first 10% of steps
EARLY_STOPPING_PATIENCE = 2   # stop if macro-F1 does not improve for 2 evals

# Precision. H100 supports bf16, which is faster and numerically stable for training.
USE_BF16 = True

# Reproducibility (the same as random_state)
SEED = 42

# Output locations
OUTPUT_DIR = "mbert_output"      # checkpoints written here during training
FINAL_MODEL_DIR = "mbert_final"  # best model saved here at the end

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

set_seed(SEED)   # make weight init and training order reproducible

# Quick device check.
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Device: Tesla T4


## 1. Load data

Same four per-language training files as the baseline, read one per language.

In [5]:
# Read the four per-language training files (tab-separated)
df_en = pd.read_csv("StereoQueerEval_EN_training.tsv", sep="\t")
df_it = pd.read_csv("StereoQueerEval_IT_training.tsv", sep="\t")
df_nl = pd.read_csv("StereoQueerEval_NL_training.tsv", sep="\t")
df_fa = pd.read_csv("StereoQueerEval_FA_training.tsv", sep="\t") # For testing purpose only

dataset_to_explore = [df_en, df_it, df_nl, df_fa]
dataset_to_train = [df_en, df_it, df_nl]
lang_names = ["english", "italian", "dutch", "persian"]
lang_names_to_train = ["english", "italian", "dutch"]

for df, name in zip(dataset_to_explore, lang_names):
    print(f"{name}: {df.shape}")

english: (2989, 7)
italian: (2400, 7)
dutch: (2238, 7)
persian: (100, 7)


## 2. Build the label

Map the stereotype column to 1 (yes) / 0 (no). The input text is built later, after the tokenizer is loaded.

In [6]:
def make_label(df):
    # 1 if the comment is a stereotype (yes), else 0.
    return (df[LABEL_COLUMN].str.lower() == POSITIVE_LABEL).astype(int)

for df in dataset_to_explore:
    df["label"] = make_label(df)

# Check class balance per language (same idea as the baseline EDA).
for df, name in zip(dataset_to_explore, lang_names):
    counts = df["label"].value_counts().to_dict()
    print(f"{name} 0(no)={counts.get(0, 0)}  1(yes)={counts.get(1, 0)}")

english 0(no)=1663  1(yes)=1326
italian 0(no)=1752  1(yes)=648
dutch 0(no)=1370  1(yes)=868
persian 0(no)=77  1(yes)=23


## 3. Split 80 / 10 / 10 per language

Split each language first, then concatenate, so every language keeps its class balance in all three sets.

Persian is for test set only

In [7]:
train_parts, val_parts, test_parts = [], [], []

for df, name in zip(dataset_to_train, lang_names):
    # 1) hold out the test set (10%)
    train_val, test = train_test_split(
        df, test_size=TEST_SIZE, random_state=SEED, stratify=df["label"]
    )
    # 2) split validation out of the remaining 90%.
    #    val must be 10% of the whole, so 0.1 / 0.9 of what is left.
    val_relative = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
    train, val = train_test_split(
        train_val, test_size=val_relative, random_state=SEED, stratify=train_val["label"]
    )
    train_parts.append(train)
    val_parts.append(val)
    test_parts.append(test)
    print(f"{name:9s} train={len(train):5d}  val={len(val):4d}  test={len(test):4d}")

# Print test length of Persian dataset
print(f"{"persian":9s} train={0:5d}  val={0:4d}  test={len(df_fa):4d}")


train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

# Append 100 Persian records to test_parts
test_df = pd.concat([test_df, df_fa], ignore_index=True)

print(f"\ntotal      train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

english   train= 2391  val= 299  test= 299
italian   train= 1919  val= 241  test= 240
dutch     train= 1790  val= 224  test= 224
persian   train=    0  val=   0  test= 100

total      train=6100  val=764  test=863


## 4. Tokenize

- Load the tokenizer that matches the model
- Build the input text
- Convert text to token IDs.

Padding is done per batch (dynamic padding) instead of padding everything to MAX_LENGTH, which is faster.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def build_text(df):
    # Join the selected text fields into one string.
    # For a single field this is just that field.
    # For several fields (later), they are joined with the tokenizer separator so the model sees field boundaries.
    parts = [df[c].fillna("").astype(str) for c in TEXT_COLUMNS]
    text = parts[0]

    # Example: "good family" + " [SEP] " + "Pride parade" = "good family [SEP] Pride parade"
    # concatenated row by row
    for p in parts[1:]:
        text = text + f" {tokenizer.sep_token} " + p
    return text

for split_df in [train_df, val_df, test_df]:
    split_df["text"] = build_text(split_df)

# Peek at one example.
print(train_df["text"].iloc[0][:200])

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

I think he is ashamed of his kid by the way he act.


In [9]:
# Keep only what the model needs: the text and the label.
keep_cols = ["text", "label"]
train_ds = Dataset.from_pandas(train_df[keep_cols], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[keep_cols], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[keep_cols], preserve_index=False)

def tokenize_function(batch):
    # Truncate to MAX_LENGTH. No padding here; the collator pads per batch.
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

# datasets.Dataset.map()
# before .map():               after .map():
#   text                         input_ids        attention_mask
#   "good family"                [101, 2204, ...]  [1, 1, 1, ...]
#   "they are sick"              [101, 2027, ...]  [1, 1, 1, ...]
train_ds = train_ds.map(tokenize_function, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize_function, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize_function, batched=True, remove_columns=["text"])

# The Trainer expects the label column to be named "labels".
train_ds = train_ds.rename_column(original_column_name="label", new_column_name="labels")
val_ds = val_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

# Dynamic padding: It pads every example in a batch up to the length of the longest example in that same batch - not to MAX_LENGTH
# Benefit: keeps batches as small as they can be, so training runs faster with no change to results.
# before padding (token counts):       after padding (all -> 9):
#   "good family"        5 tokens        5 tokens + 4 PAD
#   "they are so sick"   7 tokens        7 tokens + 2 PAD
#   "normal happy folks" 9 tokens        9 tokens + 0 PAD   <- longest in the current batch, sets the width
#
# Other batches could have different max length
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/6100 [00:00<?, ? examples/s]

Map:   0%|          | 0/764 [00:00<?, ? examples/s]

Map:   0%|          | 0/863 [00:00<?, ? examples/s]

## 5. Model

Load mBERT with a fresh 2-class classification head on top. The head starts random and is trained together with the encoder during fine-tuning.

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "no", 1: "yes"},
    label2id={"no": 0, "yes": 1},
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. Metrics

Report macro-F1 (the selection metric), plus accuracy and macro precision/recall.

In [11]:
def compute_metrics(eval_pred):
  # eval_pred: Trainer returns an object with (model outputs, true labels) bundled together
  # logits = model output/prediction; actual_labels = true_labels (y_test)
  logits, actual_labels = eval_pred
  preds = np.argmax(logits, axis=-1)  # calculate y_pred from model output
  return {
      "accuracy": accuracy_score(actual_labels, preds),
      "f1_macro": f1_score(actual_labels, preds, average="macro"),
      "precision_macro": precision_score(actual_labels, preds, average="macro", zero_division=0),
      "recall_macro": recall_score(actual_labels, preds, average="macro", zero_division=0),
  }

## 7. Training setup

Fine-tune with the Hugging Face Trainer. Evaluate every epoch, keep the best epoch by macro-F1, and stop early if it stops improving.

Can consider GridSearchCV later

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",          # When to run evaluation on the validation set (epoch=end of per epoch)
    save_strategy="epoch",          # When to save a checkpoint (a copy of the model) to disk
    logging_strategy="epoch",       # When to log the training loss
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,         # Go with metric_for_best_model; higher F1 is better
    save_total_limit=2,             # Keep only the 2 most recent checkpoints on disk
    bf16=USE_BF16,
    seed=SEED,
    report_to="none",               # set to "tensorboard" or "wandb" for live dashboards
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

## 8. Train

In [ ]:
trainer.train()

## 9. Training history

The per-epoch table (Epoch / Training Loss / Validation Loss / metrics) is printed above during training.
The same numbers are available in code for plotting or export.

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
history.head(20)

In [ ]:
# Validation macro-F1 and loss across epochs.
# Watch for the two diverging: rising loss while F1 stalls is an overfitting sign.
eval_rows = history[history["eval_f1_macro"].notna()]

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(eval_rows["epoch"], eval_rows["eval_f1_macro"], marker="o", label="val macro-F1")
ax1.set_xlabel("epoch")
ax1.set_ylabel("macro-F1")
ax2 = ax1.twinx()
ax2.plot(eval_rows["epoch"], eval_rows["eval_loss"], marker="s", color="tab:red", label="val loss")
ax2.set_ylabel("val loss")
ax1.set_title("Validation macro-F1 and loss per epoch")
plt.show()

## 10. Evaluate on the test set

Run the best model (restored by load_best_model_at_end) on the held-out test set. Same report style as the baseline.

In [ ]:
test_output = trainer.predict(test_ds)
y_true = test_output.label_ids
y_pred = np.argmax(test_output.predictions, axis=-1)

print(classification_report(y_true, y_pred, target_names=["no", "yes"]))
print("macro-F1:", f1_score(y_true, y_pred, average="macro"))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["no", "yes"])
disp.plot(cmap=plt.cm.Blues)
plt.title("mBERT - confusion matrix (test)")
plt.show()
print(cm)

## 11. Per-language results

Break the test score down by language. This shows whether the single multilingual model is weak on any one language.
Persian is tiny, so its number is noisy.

In [ ]:
# Attach predictions back to the test rows, then score each language.
test_eval = test_df.copy()
test_eval["pred"] = y_pred

# test_df was built by concatenating languages in order, so rebuild that order.
lang_of_row = []
for name, part in zip(lang_names_to_train, test_parts):
    lang_of_row.extend([name] * len(part))

# Since Persian (100 records) is for testing only, manually add to lang_of_row
lang_of_row.extend(["persian"] * 100)

test_eval["language"] = lang_of_row

for name in lang_names:
    sub = test_eval[test_eval["language"] == name]
    if len(sub) == 0:
        continue
    f1 = f1_score(sub["label"], sub["pred"], average="macro")
    print(f"{name:9s} n={len(sub):4d}  macro-F1={f1:.4f}")

## 12. Save the best model

Saves the best model and its tokenizer, so you can reload them later for inference or for predicting the competition test set.

In [ ]:
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("saved to", FINAL_MODEL_DIR)